# MNIST MultiScreen Screening Trace

This notebook loads a MNIST-trained `MultiScreenForImageClassification` checkpoint, enables `GatedScreening` tracing, and visualizes the saved screening relevance matrix plus the screened vectors before and after the gate.

Edit `CHECKPOINT_PATH` in the next cell to point at a checkpoint saved by `train/mnist.py`.

In [7]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "src" / "screening").exists():
            return path
    raise RuntimeError("Could not find project root")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import ipywidgets as widgets
import torch
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from screening import MultiScreenForImageClassification
from train.mnist import TrainConfig, build_loaders, image_logits, load_config


CONFIG_PATH = PROJECT_ROOT / "configs" / "mnist.yaml"
CHECKPOINT_PATH = None  # Example: PROJECT_ROOT / "checkpoints" / "mnist.pt"

SAMPLE_INDEX = 0
LAYER_INDEX = -1
HEAD_INDEX = None  # None means average heads. Use -2 to show all heads.
QUERY_PATCH = None  # None means use the query patch with the largest outgoing score

In [8]:
cfg = load_config(CONFIG_PATH)
checkpoint_path = (
    Path(CHECKPOINT_PATH) if CHECKPOINT_PATH is not None else cfg.checkpoint_path
)
if checkpoint_path is not None and not checkpoint_path.is_absolute():
    checkpoint_path = PROJECT_ROOT / checkpoint_path

checkpoint = None
if checkpoint_path is not None and checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    if "config" in checkpoint:
        cfg = TrainConfig.model_validate(checkpoint["config"])
    display(Markdown(f"Loaded checkpoint: `{checkpoint_path}`"))
else:
    display(
        Markdown(
            "No checkpoint loaded. The model will use random weights until `CHECKPOINT_PATH` is set."
        )
    )

data_dir = cfg.data_dir if cfg.data_dir.is_absolute() else PROJECT_ROOT / cfg.data_dir
cfg = cfg.model_copy(update={"data_dir": data_dir})

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiScreenForImageClassification(
    hidden_dim=cfg.hidden_dim,
    num_heads=cfg.num_heads,
    num_blocks=cfg.num_blocks,
    num_classes=10,
    in_channels=1,
    patch_size=cfg.patch_size,
    window_threshold=cfg.window_threshold,
).to(device)
if checkpoint is not None:
    model.load_state_dict(checkpoint["model"])

model.eval()
model.set_trace_screening(True)
display(
    Markdown(
        f"Device: `{device}` | hidden_dim={cfg.hidden_dim}, heads={cfg.num_heads}, blocks={cfg.num_blocks}, patch_size={cfg.patch_size}"
    )
)

Loaded checkpoint: `/media/plat/WDC20/Documents/python/screening/output/mnist-05.ckpt`

Device: `cuda` | hidden_dim=64, heads=8, blocks=8, patch_size=2

In [9]:
_train_loader, val_loader = build_loaders(cfg)
val_dataset = val_loader.dataset
display(Markdown(f"Loaded `{len(val_dataset)}` validation images for inspection."))

Loaded `10000` validation images for inspection.

In [10]:
def to_numpy(values: torch.Tensor):
    return values.detach().float().cpu().numpy()


def mnist_pixels(image: torch.Tensor) -> torch.Tensor:
    return (image[0, 0].detach().cpu() * 0.3081 + 0.1307).clamp(0, 1)


def patch_grid_shape(image: torch.Tensor, patch_size: int) -> tuple[int, int]:
    height, width = image.shape[-2:]
    if height < patch_size or width < patch_size:
        raise ValueError(
            f"patch_size={patch_size} is larger than image size {height}x{width}"
        )
    return (height - patch_size) // patch_size + 1, (
        width - patch_size
    ) // patch_size + 1


def hide_ticks(ax) -> None:
    ax.set_xticks([])
    ax.set_yticks([])


def plot_maps(maps, *, figsize=None):
    if figsize is None:
        figsize = (3.0 * len(maps), 3.2)
    fig, axes = plt.subplots(1, len(maps), figsize=figsize, constrained_layout=True)
    if len(maps) == 1:
        axes = [axes]

    for ax, (title, values, cmap, show_colorbar) in zip(axes, maps):
        im = ax.imshow(to_numpy(values), cmap=cmap, interpolation="nearest")
        ax.set_title(title)
        hide_ticks(ax)
        if show_colorbar:
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return fig


def clamp_int(value: int, low: int, high: int) -> int:
    return max(low, min(int(value), high))


def head_label(head_index: int) -> str:
    if head_index == -2:
        return "all heads"
    return "mean" if head_index < 0 else f"head {head_index}"


def evaluate_sample(sample_index: int):
    sample_index = clamp_int(sample_index, 0, len(val_dataset) - 1)
    image, label = val_dataset[sample_index]
    image = image.unsqueeze(0).to(device)

    model.set_trace_screening(True)
    with torch.no_grad():
        logits = image_logits(model, image, cfg.patch_size)
        probs = logits.softmax(dim=-1)[0].detach().cpu()
    pred = int(probs.argmax().item())
    return sample_index, image, int(label), probs, pred


def layer_trace(layer_index: int):
    layer_index = clamp_int(layer_index, 0, len(model.layers) - 1)
    trace = model.layers[layer_index].screening_trace
    assert trace, (
        "No screening trace captured. Did you call model.set_trace_screening(True)?"
    )
    return layer_index, trace


def trace_views(
    image: torch.Tensor,
    layer_index: int,
    head_index: int,
    query_patch: int,
) -> dict[str, torch.Tensor | int | str]:
    layer_index, trace = layer_trace(layer_index)
    score = trace["score"][0]
    before_gate = trace["before_gate"][0]
    after_gate = trace["after_gate"][0]

    if head_index < 0:
        selected_score = score.mean(dim=0)
        before_norm = before_gate.norm(dim=-1).mean(dim=0)
        after_norm = after_gate.norm(dim=-1).mean(dim=0)
    else:
        head_index = clamp_int(head_index, 0, score.size(0) - 1)
        selected_score = score[head_index]
        before_norm = before_gate[head_index].norm(dim=-1)
        after_norm = after_gate[head_index].norm(dim=-1)

    grid_h, grid_w = patch_grid_shape(image, cfg.patch_size)
    patch_count = grid_h * grid_w
    if query_patch < 0:
        query_patch = int(selected_score.sum(dim=-1).argmax().item())
    query_patch = clamp_int(query_patch, 0, patch_count - 1)

    key_mass = selected_score.mean(dim=0).view(grid_h, grid_w)
    query_mass = selected_score.mean(dim=1).view(grid_h, grid_w)
    before_norm = before_norm.view(grid_h, grid_w)
    after_norm = after_norm.view(grid_h, grid_w)
    gate_ratio = after_norm / before_norm.clamp_min(1e-12)
    query_to_keys = selected_score[query_patch].view(grid_h, grid_w)

    return {
        "layer_index": layer_index,
        "head_index": head_index,
        "query_patch": query_patch,
        "query_row": query_patch // grid_w,
        "query_col": query_patch % grid_w,
        "query_mass": query_mass,
        "key_mass": key_mass,
        "before_norm": before_norm,
        "after_norm": after_norm,
        "gate_ratio": gate_ratio,
        "query_to_keys": query_to_keys,
    }


def plot_all_head_maps(head_views: list[dict[str, torch.Tensor | int | str]]):
    columns = [
        ("query mass", "query_mass"),
        ("key mass", "key_mass"),
        ("before gate norm", "before_norm"),
        ("after gate norm", "after_norm"),
        ("after / before", "gate_ratio"),
        ("query -> keys", "query_to_keys"),
    ]
    nrows = len(head_views)
    ncols = len(columns)
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(2.45 * ncols, 1.65 * nrows),
        constrained_layout=True,
        squeeze=False,
    )

    images = []
    for col, (title, key) in enumerate(columns):
        values = torch.stack([view[key].float() for view in head_views])
        vmin = float(values.min().item())
        vmax = float(values.max().item())
        if vmin == vmax:
            vmax = vmin + 1e-12

        column_image = None
        for row, view in enumerate(head_views):
            ax = axes[row, col]
            column_image = ax.imshow(
                to_numpy(view[key]),
                cmap="magma",
                interpolation="nearest",
                vmin=vmin,
                vmax=vmax,
            )
            if row == 0:
                ax.set_title(title)
            if col == 0:
                ax.set_ylabel(
                    f"head {view['head_index']}\nq={view['query_patch']}",
                    rotation=0,
                    labelpad=34,
                    va="center",
                )
            hide_ticks(ax)
        images.append(column_image)

    for col, image in enumerate(images):
        fig.colorbar(image, ax=axes[:, col], fraction=0.025, pad=0.01)
    return fig


def layer_summary_rows() -> list[dict[str, float | int]]:
    rows = []
    for index, traced_layer in enumerate(model.layers):
        trace = traced_layer.screening_trace
        if not trace:
            continue
        traced_score = trace["score"][0]
        traced_before = trace["before_gate"][0]
        traced_after = trace["after_gate"][0]
        before_mean = traced_before.norm(dim=-1).mean().item()
        after_mean = traced_after.norm(dim=-1).mean().item()
        rows.append(
            {
                "layer": index,
                "score_mean": traced_score.mean().item(),
                "score_max": traced_score.max().item(),
                "before_gate_norm": before_mean,
                "after_gate_norm": after_mean,
                "gate_ratio": after_mean / max(before_mean, 1e-12),
            }
        )
    return rows


def display_layer_summary() -> None:
    rows = layer_summary_rows()
    if not rows:
        display(Markdown("No screening trace captured yet."))
        return

    table = [
        "| layer | score mean | score max | before norm | after norm | gate ratio |",
        "|---:|---:|---:|---:|---:|---:|",
    ]
    for row in rows:
        table.append(
            f"| {row['layer']:02d} | {row['score_mean']:.4f} | {row['score_max']:.4f} | "
            f"{row['before_gate_norm']:.4f} | {row['after_gate_norm']:.4f} | "
            f"{row['gate_ratio']:.4f} |"
        )
    display(Markdown("\n".join(table)))


def render_trace(
    sample_index: int,
    layer_index: int,
    head_index: int,
    query_patch: int,
    show_summary: bool,
) -> None:
    sample_index, image, label, probs, pred = evaluate_sample(sample_index)
    if head_index == -2:
        layer_index = clamp_int(layer_index, 0, len(model.layers) - 1)
        head_views = [
            trace_views(image, layer_index, head, query_patch)
            for head in range(cfg.num_heads)
        ]
        query_text = "auto per head" if query_patch < 0 else str(query_patch)
        display(
            Markdown(
                f"Sample: **{sample_index}** | Label: **{label}** | Prediction: **{pred}** | "
                f"confidence={float(probs[pred]):.3f} | Layer: **{layer_index}** | "
                f"Head: **all heads** | Query: **{query_text}**"
            )
        )
        plot_maps(
            [("input", mnist_pixels(image), "gray", False)],
            figsize=(2.6, 2.6),
        )
        plt.show()
        plot_all_head_maps(head_views)
        plt.show()
        if show_summary:
            display_layer_summary()
        return

    views = trace_views(image, layer_index, head_index, query_patch)
    layer_index = int(views["layer_index"])
    query_patch = int(views["query_patch"])

    display(
        Markdown(
            f"Sample: **{sample_index}** | Label: **{label}** | Prediction: **{pred}** | "
            f"confidence={float(probs[pred]):.3f} | Layer: **{layer_index}** | "
            f"Head: **{head_label(head_index)}**"
        )
    )
    plot_maps(
        [
            ("input", mnist_pixels(image), "gray", False),
            ("query mass", views["query_mass"], "magma", True),
            ("key mass", views["key_mass"], "magma", True),
            ("before gate norm", views["before_norm"], "magma", True),
            ("after gate norm", views["after_norm"], "magma", True),
            ("after / before", views["gate_ratio"], "magma", True),
        ],
        figsize=(18.0, 3.2),
    )
    plt.show()

    display(
        Markdown(
            f"Query patch: **{query_patch}** at "
            f"row={views['query_row']}, col={views['query_col']}"
        )
    )
    plot_maps(
        [
            ("input", mnist_pixels(image), "gray", False),
            ("query -> keys", views["query_to_keys"], "magma", True),
        ],
        figsize=(6.5, 3.2),
    )
    plt.show()

    if show_summary:
        display_layer_summary()

In [11]:
preview_image, _ = val_dataset[0]
grid_h, grid_w = patch_grid_shape(preview_image, cfg.patch_size)
patch_count = grid_h * grid_w

initial_layer = LAYER_INDEX if LAYER_INDEX >= 0 else len(model.layers) + LAYER_INDEX
initial_layer = clamp_int(initial_layer, 0, len(model.layers) - 1)
initial_head = (
    -1 if HEAD_INDEX is None else clamp_int(HEAD_INDEX, -2, cfg.num_heads - 1)
)
initial_query = (
    -1 if QUERY_PATCH is None else clamp_int(QUERY_PATCH, 0, patch_count - 1)
)

style = {"description_width": "90px"}
sample_control = widgets.BoundedIntText(
    value=clamp_int(SAMPLE_INDEX, 0, len(val_dataset) - 1),
    min=0,
    max=len(val_dataset) - 1,
    description="sample",
    style=style,
)
layer_control = widgets.IntSlider(
    value=initial_layer,
    min=0,
    max=len(model.layers) - 1,
    step=1,
    description="layer",
    continuous_update=False,
    style=style,
)
head_control = widgets.Dropdown(
    options=[
        ("mean", -1),
        ("all heads", -2),
        *[(f"head {index}", index) for index in range(cfg.num_heads)],
    ],
    value=initial_head,
    description="head",
    style=style,
)
query_control = widgets.IntSlider(
    value=initial_query,
    min=-1,
    max=patch_count - 1,
    step=1,
    description="query",
    continuous_update=False,
    style=style,
)
summary_control = widgets.Checkbox(
    value=False,
    description="summary",
    indent=False,
)

controls = {
    "sample_index": sample_control,
    "layer_index": layer_control,
    "head_index": head_control,
    "query_patch": query_control,
    "show_summary": summary_control,
}
output = widgets.interactive_output(render_trace, controls)

display(
    Markdown(
        "Set `query=-1` to select the strongest outgoing query patch automatically. "
        "Choose `all heads` to compare every head vertically."
    )
)
display(
    widgets.VBox(
        [
            widgets.HBox([sample_control, layer_control]),
            widgets.HBox([head_control, query_control, summary_control]),
            output,
        ]
    )
)

Set `query=-1` to select the strongest outgoing query patch automatically. Choose `all heads` to compare every head vertically.

In [12]:
display_layer_summary()

| layer | score mean | score max | before norm | after norm | gate ratio |
|---:|---:|---:|---:|---:|---:|
| 00 | 0.0888 | 0.9871 | 0.7791 | 0.3227 | 0.4142 |
| 01 | 0.0373 | 0.9869 | 0.6975 | 0.3382 | 0.4849 |
| 02 | 0.0800 | 0.9803 | 0.6181 | 0.2631 | 0.4256 |
| 03 | 0.0076 | 0.9550 | 0.4734 | 0.2174 | 0.4591 |
| 04 | 0.0187 | 0.9937 | 0.4789 | 0.2552 | 0.5330 |
| 05 | 0.0178 | 0.9673 | 0.4838 | 0.2879 | 0.5950 |
| 06 | 0.0091 | 0.9617 | 0.5431 | 0.3003 | 0.5530 |
| 07 | 0.0373 | 0.9753 | 0.7859 | 0.5207 | 0.6626 |